In [0]:
-- EXPLORING THE DATASETS 
  
   --1. How many user profiles do we have? 5 375 subscribers   
   SELECT COUNT(userid) AS total_subscribers
   FROM workspace.bright_learn.bright_tv_user_profiles;

   --2. How many Channels do we have? 21 Channels

    SELECT DISTINCT(Channel2) Total_Channels
   FROM workspace.bright_learn.bright_tv_viewership;



SELECT 

    -- Viewership Data with Null Handling
  v.UserID,
  COALESCE(v.Channel2, 'Unknown Channel') AS Channel,
  COALESCE(v.`Duration 2`, '00:02:00') AS Duration,

  -- Date and Time Conversions (UTC to SA Time: UTC+2)
  -- 1. Convert the string to a timestamp and shift to Africa/Johannesburg
  FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg') AS SATimestamp,
  
  -- 2. Separate Date and Time
  DATE(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg')) AS RecordDate,
  DATE_FORMAT(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg'), 'HH:mm') AS RecordTime,
  
  -- 3. Day Name Column
  DATE_FORMAT(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg'), 'EEEE') AS DayName,

  -- 4. Time Class Categorisation
  CASE
    WHEN HOUR(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg')) BETWEEN 6 AND 11 THEN 'Morning'
    WHEN HOUR(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg')) BETWEEN 12 AND 17 THEN 'Afternoon'
    WHEN HOUR(FROM_UTC_TIMESTAMP(TO_TIMESTAMP(v.RecordDate2, 'yyyy/MM/dd HH:mm'), 'Africa/Johannesburg')) BETWEEN 18 AND 23 THEN 'Evening'
    ELSE 'Night'
  END AS DayPart,

  -- Channel Category Logic
  CASE
    WHEN v.Channel2 IN ('ICC Cricket World Cup 2011', 'Supersport Live Events', 'SuperSport Blitz', 'Live on SuperSport', 'Wimbledon') THEN 'Sports'
    WHEN v.Channel2 IN ('Cartoon Network', 'Boomerang') THEN 'Kids'
    WHEN v.Channel2 = 'CNN' THEN 'News'
    WHEN v.Channel2 IN ('Trace TV', 'E! Entertainment', 'SawSee', 'Channel O', 'Africa Magic', 'kykNET', 'Vuzu', 'M-Net', 'MK') THEN 'Entertainment'
    WHEN v.Channel2 IN ('Break in transmission', 'DStv Events 1') THEN 'Technical/Other'
    ELSE 'Uncategorized'
  END AS ChannelCategory,

  -- User Profile Data with Null Handling
  COALESCE(NULLIF(p.Name, 'None'), 'Unknown') AS Name,
  COALESCE(NULLIF(p.Surname, 'None'), 'Unknown') AS Surname,
  COALESCE(NULLIF(p.Gender, 'None'), 'Not Specified') AS Gender,
  COALESCE(NULLIF(p.Age, 0), NULL) AS Age,

  -- Age Group Categorisation
  CASE
    WHEN p.Age = 0 OR p.Age IS NULL THEN 'Unknown'
    WHEN p.Age < 20 THEN 'Teen/Child'
    WHEN p.Age BETWEEN 20 AND 35 THEN 'Young Adult'
    WHEN p.Age BETWEEN 36 AND 55 THEN 'Adult'
    WHEN p.Age > 55 THEN 'Senior'
    ELSE 'Unknown'
  END AS AgeGroup


FROM workspace.bright_learn.bright_tv_viewership v
LEFT JOIN workspace.bright_learn.bright_tv_user_profiles p ON v.UserID = p.UserID;